# 은행 마케팅: 누수 방지 및 재현 가능한 최적화
전화 직전 가입 가능성 예측. `duration`은 항상 제외합니다.
원본 URL의 **교육용 데이터만** 사용하며, UCI 전체 데이터의 정답으로 제출 행을 복원하지 않습니다.

## 실행 방법
이 노트북, `input_bank_submission.csv`, `bank-full.csv`를 같은 폴더에 두고 Run All을 실행하세요.
Python 3.12, numpy, pandas, scikit-learn, threadpoolctl이 필요합니다. 정확한 버전은 함께 제공한 requirements.txt 참고.
학습 CSV가 없으면 원본 URL에서 다운로드합니다. 데이터 SHA-256을 기록합니다.

## 원본에서 확인한 사항
- duration 제외 및 학습셋에서 계산한 최빈값 사용은 올바른 방향입니다.
- 무작위 85/15 단일 분할, stratify 없음; LR 스케일링 없음; 기본 RF 100개 트리 사용.
- 검증에서 threshold=0.4를 시험했지만 제출은 predict()의 기본 0.5 사용.
- 제출 데이터에는 job/education unknown 최빈값 치환이 빠져 학습과 전처리가 다릅니다.
- 교차검증·체계적 튜닝·전체 데이터 재학습이 없습니다. 저장된 점수는 현재 실험과 분할이 달라 직접 비교할 수 없습니다.

## 검증 설계
모델 입력이 동일한 행은 같은 그룹에 넣습니다. 5분할 중 하나(약 20%)를 최종 홀드아웃으로 잠그고,
나머지에서 StratifiedGroupKFold 3분할로 AP(average precision)를 비교합니다.
최종 홀드아웃은 모델·임계값을 고른 후 한 번만 평가합니다.
CV 최고 점수 및 선택 후 OOF 점수는 선택 편향이 있으므로 최종 일반화 성능은 홀드아웃으로 판단합니다.
고객 ID가 없어 서로 다른 기록의 동일 고객을 완전히 분리할 수는 없습니다.


In [1]:
import os, sys, json, time, hashlib, platform
from pathlib import Path
from io import StringIO
from urllib.request import urlopen
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score, balanced_accuracy_score, confusion_matrix)
from threadpoolctl import threadpool_limits
SEED = 42
np.random.seed(SEED)
START = time.perf_counter()
thread_limits = threadpool_limits(limits=4)
OUT = Path('.')
versions = {'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'scikit-learn': sklearn.__version__}
print(versions)


{'python': '3.15.0', 'numpy': '2.5.4', 'pandas': '3.0.6', 'scikit-learn': '1.9.1'}


In [2]:
URL = 'https://raw.githubusercontent.com/suyongzz/AICampus/refs/heads/main/bank-full.csv'
data_path = Path('bank-full.csv')
if not data_path.exists():
    with urlopen(URL, timeout=60) as response:
        data_path.write_bytes(response.read())
raw = data_path.read_bytes()
sha256 = hashlib.sha256(raw).hexdigest()
text = raw.decode('utf-8-sig')
df = pd.read_csv(StringIO(text), sep=';')
if df.shape[1] == 1:
    wrapped = pd.read_csv(StringIO(text))
    text = '\n'.join([wrapped.columns[0]] + wrapped.iloc[:, 0].astype(str).tolist())
    df = pd.read_csv(StringIO(text), sep=';')
submit = pd.read_csv('input_bank_submission.csv')
assert df.shape[1] == 17 and set(df.y.unique()) == {'yes', 'no'}
assert 'y' not in submit.columns, '제출 입력에 정답 열이 있습니다. 확인 필요.'
raw_features = df.columns.drop('y').tolist()
assert set(submit.columns) == set(raw_features), '제출 입력 스키마 확인 필요'
print('학습 / 제출:', df.shape, submit.shape, 'SHA256:', sha256)
print('가입률:', df.y.eq('yes').mean(), '결측값:', int(df.isna().sum().sum()))
print('완전 중복:', int(df.duplicated().sum()))
# 제출과 완전히 같은 입력 행은 학습에서 제외 (타깃은 비교에 쓰지 않음).
train_hash = pd.util.hash_pandas_object(df[raw_features], index=False)
submit_hash = pd.util.hash_pandas_object(submit[raw_features], index=False)
overlap = train_hash.isin(submit_hash)
print('제출과 동일한 학습 입력 제거:', int(overlap.sum()))
df = df.loc[~overlap].reset_index(drop=True)


학습 / 제출: (45111, 17) (100, 16) SHA256: 7bc620b882f5e55adfe0c222a51eabcf340cbeb2cca68f9947243c888cb78f8e
가입률: 0.11618008911351999 결측값: 0
완전 중복: 0
제출과 동일한 학습 입력 제거: 0


## 전처리와 변수의 이용 가능 시점
`unknown`은 정보가 없는 상태 자체를 나타내는 범주로 보존합니다. 실제 결측만 학습 fold의 최빈값/중앙값으로 보완합니다.
`pdays=-1`은 이전 연락 없음 표시와 NaN으로 나눕니다. 음수 balance나 큰 값을 임의로 삭제하지 않습니다.
모든 통계적 전처리는 Pipeline 안에서 학습합니다. LR에만 스케일링을 적용합니다.
`contact/day/month/campaign`은 해당 연락 계획과 현재까지의 연락 횟수를 예측 시점에 안다는 가정입니다.
캠페인 시작 전 일괄 선정에서는 이 가정이 성립하지 않을 수 있으므로 네 변수를 추가로 제외하여 재검증해야 합니다.


In [3]:
def make_features(frame):
    x = frame.drop(columns=['y', 'duration'], errors='ignore').copy()
    x['previously_contacted'] = x['pdays'].ne(-1).astype(int)
    x['pdays'] = x['pdays'].replace(-1, np.nan)
    return x
X, X_submit = make_features(df), make_features(submit)
y = df.y.eq('yes').astype(int)
assert X.columns.equals(X_submit.columns)
assert 'duration' not in X and 'y' not in X
groups = pd.util.hash_pandas_object(X, index=False).to_numpy()
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
dev_idx, hold_idx = next(splitter.split(X, y, groups))
X_dev, y_dev = X.iloc[dev_idx], y.iloc[dev_idx]
X_hold, y_hold = X.iloc[hold_idx], y.iloc[hold_idx]
dev_groups = groups[dev_idx]
assert not set(dev_groups).intersection(groups[hold_idx])
cv = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED + 1)
          .split(X_dev, y_dev, dev_groups))
for a, b in cv:
    assert not set(dev_groups[a]).intersection(dev_groups[b])
cats = X.select_dtypes(exclude='number').columns.tolist()
nums = X.select_dtypes(include='number').columns.tolist()
def pipeline(model, scale=False):
    numeric = [('impute', SimpleImputer(strategy='median'))]
    if scale: numeric.append(('scale', StandardScaler()))
    preprocessing = ColumnTransformer([
        ('num', Pipeline(numeric), nums),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                          ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cats)
    ])
    return Pipeline([('preprocess', preprocessing), ('model', model)])
print('개발 / 최종 홀드아웃:', len(dev_idx), len(hold_idx))


개발 / 최종 홀드아웃: 36088 9023


## 제한된 예산의 모델·하이퍼파라미터 탐색
LR 2종, 원본과 같은 기본 RF(공통 전처리 사용), 히스토그램 부스팅 3종을 같은 fold에서 비교합니다.
6후보 × 3fold이며 모든 모델은 한 번에 하나씩 학습하고 스레드는 4개로 제한합니다.
AP 최고 모델을 선택합니다. ROC-AUC·F1·정확도와 fit 시간도 저장합니다.
AP는 여기서 average precision이며 사다리꼴 적분 PR-AUC와 같지 않습니다.


In [4]:
candidates = {
    'LR_C0.1': pipeline(LogisticRegression(C=0.1, max_iter=1000, random_state=SEED), True),
    'LR_C1': pipeline(LogisticRegression(C=1, max_iter=1000, random_state=SEED), True),
    'RF_original_params': pipeline(RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=4)),
    'HGB_small': pipeline(HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08,
        max_leaf_nodes=15, l2_regularization=5, early_stopping=False, random_state=SEED)),
    'HGB_medium': pipeline(HistGradientBoostingClassifier(max_iter=250, learning_rate=0.07,
        max_leaf_nodes=31, l2_regularization=10, early_stopping=False, random_state=SEED)),
    'HGB_regularized': pipeline(HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
        max_leaf_nodes=15, min_samples_leaf=40, l2_regularization=20,
        early_stopping=False, random_state=SEED)),
}
scoring = {'ap':'average_precision', 'roc_auc':'roc_auc', 'accuracy':'accuracy', 'f1':'f1'}
rows, fold_rows = [], []
for name, estimator in candidates.items():
    scores = cross_validate(estimator, X_dev, y_dev, cv=cv, scoring=scoring,
                            n_jobs=1, return_train_score=True, error_score='raise')
    row = {'model':name, 'fit_seconds_mean':float(scores['fit_time'].mean())}
    for metric in scoring:
        row[metric+'_mean'] = float(scores['test_'+metric].mean())
        row[metric+'_std'] = float(scores['test_'+metric].std(ddof=1))
    row['train_ap_mean'] = float(scores['train_ap'].mean())
    rows.append(row)
    for i in range(len(cv)):
        fold_rows.append({'model':name,'fold':i+1,
            **{key:float(value[i]) for key,value in scores.items()}})
    print(name, 'AP=', round(row['ap_mean'], 5), 'fit seconds=', round(row['fit_seconds_mean'], 2), flush=True)
results = pd.DataFrame(rows).sort_values('ap_mean', ascending=False)
results.to_csv('cv_results.csv', index=False)
pd.DataFrame(fold_rows).to_csv('cv_folds.csv', index=False)
best_name = results.iloc[0]['model']
best = candidates[best_name]
print(results.to_string(index=False))


LR_C0.1 AP= 0.40096 fit seconds= 0.1
LR_C1 AP= 0.40068 fit seconds= 0.08
RF_original_params AP= 0.41268 fit seconds= 0.43
HGB_small AP= 0.44522 fit seconds= 0.37
HGB_medium AP= 0.44264 fit seconds= 0.66
HGB_regularized AP= 0.44816 fit seconds= 0.54
             model  fit_seconds_mean  ap_mean   ap_std  roc_auc_mean  roc_auc_std  accuracy_mean  accuracy_std  f1_mean   f1_std  train_ap_mean
   HGB_regularized          0.542653 0.448157 0.004009      0.801789     0.004632       0.894342      0.000579 0.348010 0.008124       0.525994
         HGB_small          0.368912 0.445224 0.002019      0.799463     0.006258       0.894452      0.000123 0.348413 0.012045       0.556397
        HGB_medium          0.655566 0.442643 0.003049      0.798796     0.006936       0.894286      0.000168 0.355816 0.009281       0.624806
RF_original_params          0.429326 0.412681 0.003438      0.778662     0.004494       0.893399      0.001483 0.332625 0.014007       1.000000
           LR_C0.1          0.0

## 임계값 결정 및 최종 홀드아웃
정답 불균형 때문에 정확도만 최대화하면 가입 고객을 놓칠 수 있습니다.
마케팅 비용/이익이나 대회 채점 지표가 주어지지 않아 기본 제출은 **개발 OOF F1 최대화** 임계값을 사용합니다.
0.05~0.60의 고정 격자에서 결정하며 홀드아웃은 사용하지 않습니다. 0.5 제출 파일도 별도로 제공합니다.
0.5는 정확도 최적 임계값이라는 보장은 없습니다. 실제 비용이 있다면 임계값을 다시 설계해야 합니다.


In [5]:
oof = cross_val_predict(best, X_dev, y_dev, cv=cv, method='predict_proba', n_jobs=1)[:,1]
grid = np.linspace(0.05, 0.60, 112)
threshold_scores = np.array([f1_score(y_dev, oof >= t) for t in grid])
threshold = float(grid[np.argmax(threshold_scores)])
pd.DataFrame({'threshold':grid, 'oof_f1':threshold_scores}).to_csv('threshold_search.csv', index=False)
def metrics(target, probability, cutoff):
    pred = probability >= cutoff
    k = max(1, int(np.ceil(len(target)*0.1)))
    top = np.argsort(-probability, kind='stable')[:k]
    return {'ap':float(average_precision_score(target, probability)),
        'roc_auc':float(roc_auc_score(target, probability)),
        'accuracy':float(accuracy_score(target,pred)),
        'balanced_accuracy':float(balanced_accuracy_score(target,pred)),
        'precision':float(precision_score(target,pred,zero_division=0)),
        'recall':float(recall_score(target,pred,zero_division=0)),
        'f1':float(f1_score(target,pred,zero_division=0)),
        'precision_at_top10pct':float(np.asarray(target)[top].mean()),
        'confusion_matrix_no_yes':confusion_matrix(target,pred,labels=[0,1]).tolist()}
fitted = clone(best).fit(X_dev,y_dev)
hold_probability = fitted.predict_proba(X_hold)[:,1]
hold_metrics = metrics(y_hold, hold_probability, threshold)
hold_default = metrics(y_hold, hold_probability, 0.5)
print('선택 모델:',best_name,'OOF F1 임계값:',threshold)
print('최종 홀드아웃 (OOF 임계값):',json.dumps(hold_metrics,ensure_ascii=False,indent=2))
print('최종 홀드아웃 (0.5):',json.dumps(hold_default,ensure_ascii=False,indent=2))
print('모두 no 정확도:',float(1-y_hold.mean()))
pd.DataFrame({'source_row':hold_idx,'y_true':y_hold.to_numpy(),
              'probability_yes':hold_probability}).to_csv('holdout_predictions.csv',index=False)


선택 모델: HGB_regularized OOF F1 임계값: 0.20360360360360358
최종 홀드아웃 (OOF 임계값): {
  "ap": 0.4514874020457789,
  "roc_auc": 0.7899371121062423,
  "accuracy": 0.8737670397872105,
  "balanced_accuracy": 0.7145730774684071,
  "precision": 0.4610051993067591,
  "recall": 0.5071496663489037,
  "f1": 0.4829777576032683,
  "precision_at_top10pct": 0.5138427464008859,
  "confusion_matrix_no_yes": [
    [
      7352,
      622
    ],
    [
      517,
      532
    ]
  ]
}
최종 홀드아웃 (0.5): {
  "ap": 0.4514874020457789,
  "roc_auc": 0.7899371121062423,
  "accuracy": 0.8930510916546603,
  "balanced_accuracy": 0.6083383364858574,
  "precision": 0.6014492753623188,
  "recall": 0.23736892278360344,
  "f1": 0.34039644565960353,
  "precision_at_top10pct": 0.5138427464008859,
  "confusion_matrix_no_yes": [
    [
      7809,
      165
    ],
    [
      800,
      249
    ]
  ]
}
모두 no 정확도: 0.8837415493738224


## 행 순서에 따른 이동 점검 (보조 진단)
UCI는 원본 bank-full.csv가 날짜 순이라고 설명합니다. 교육용 파일에도 순서가 유지되었다는 가정하에
앞 80%에서 학습하고 뒤 20%를 예측합니다. 동일 입력 그룹은 학습 쪽에서 제거합니다.
이 검사는 별도 시간 일반화 보장이 아닙니다. 실제 날짜/고객 ID가 없고, 모델 선택에 사용한 행이 포함되어
있으므로 튜닝에서 완전히 독립적인 시간 검증으로 해석하면 안 됩니다. 결과로 모델·임계값을 변경하지 않습니다.
실제 배포에는 날짜와 고객 ID를 확보한 전진 검증이 필요합니다.
출처: https://archive.ics.uci.edu/dataset/222/bank+marketing


In [6]:
cut = int(len(X)*0.8)
late_idx = np.arange(cut,len(X))
early_idx = np.arange(cut)
early_idx = early_idx[~np.isin(groups[early_idx],groups[late_idx])]
temporal_model = clone(best).fit(X.iloc[early_idx],y.iloc[early_idx])
temporal_probability = temporal_model.predict_proba(X.iloc[late_idx])[:,1]
temporal_metrics = metrics(y.iloc[late_idx],temporal_probability,threshold)
print('행 순서 보조 진단:',json.dumps(temporal_metrics,ensure_ascii=False,indent=2))


행 순서 보조 진단: {
  "ap": 0.49593278358694093,
  "roc_auc": 0.6970475568400304,
  "accuracy": 0.7053086556577635,
  "balanced_accuracy": 0.606996620505124,
  "precision": 0.5426659230340212,
  "recall": 0.3460170697012802,
  "f1": 0.4225841476655809,
  "precision_at_top10pct": 0.5105204872646734,
  "confusion_matrix_no_yes": [
    [
      5391,
      820
    ],
    [
      1839,
      973
    ]
  ]
}


## 전체 재학습 및 제출
선택한 설정을 바꾸지 않고 전체 허용 학습 데이터로 재학습합니다. 제출 입력의 행 순서를 그대로 보존하고
원본 제출 코드와 같은 `y` 한 열에 yes/no를 저장합니다. 확률은 별도 진단 파일입니다.
제출 정답이 없으므로 제출 데이터 정확도는 측정할 수 없습니다.


In [7]:
final_model = clone(best).fit(X,y)
submit_probability = final_model.predict_proba(X_submit)[:,1]
predictions = pd.DataFrame({'y':np.where(submit_probability >= threshold,'yes','no')})
predictions.to_csv('bank_predictions.csv',index=False)
pd.DataFrame({'y':np.where(submit_probability >= 0.5,'yes','no')}).to_csv('bank_predictions_05.csv',index=False)
pd.DataFrame({'input_row_0based':np.arange(len(submit)), 'probability_yes':submit_probability,
              'prediction':predictions.y}).to_csv('submission_probabilities.csv',index=False)
check = pd.read_csv('bank_predictions.csv')
assert check.shape == (len(submit),1) and check.columns.tolist() == ['y']
assert set(check.y) <= {'yes','no'} and not check.isna().any().any()
summary = {'source_url':URL,'sha256':sha256,'versions':versions,'seed':SEED,
    'train_rows':len(X),'submission_rows':len(submit),'positive_rate':float(y.mean()),
    'removed_submission_overlap':int(overlap.sum()), 'best_model':best_name,'threshold':threshold,
    'holdout':hold_metrics,'holdout_threshold_05':hold_default,
    'row_order_diagnostic':temporal_metrics,'total_seconds':time.perf_counter()-START,
    'submission_counts':check.y.value_counts().to_dict()}
Path('run_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
Path('requirements.txt').write_text('\n'.join(f'{key}=={value}' for key,value in versions.items() if key!='python')+'\nthreadpoolctl>=3.1\n',encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))


{
  "source_url": "https://raw.githubusercontent.com/suyongzz/AICampus/refs/heads/main/bank-full.csv",
  "sha256": "7bc620b882f5e55adfe0c222a51eabcf340cbeb2cca68f9947243c888cb78f8e",
  "versions": {
    "python": "3.15.0",
    "numpy": "2.5.4",
    "pandas": "3.0.6",
    "scikit-learn": "1.9.1"
  },
  "seed": 42,
  "train_rows": 45111,
  "submission_rows": 100,
  "positive_rate": 0.11618008911351999,
  "removed_submission_overlap": 0,
  "best_model": "HGB_regularized",
  "threshold": 0.20360360360360358,
  "holdout": {
    "ap": 0.4514874020457789,
    "roc_auc": 0.7899371121062423,
    "accuracy": 0.8737670397872105,
    "balanced_accuracy": 0.7145730774684071,
    "precision": 0.4610051993067591,
    "recall": 0.5071496663489037,
    "f1": 0.4829777576032683,
    "precision_at_top10pct": 0.5138427464008859,
    "confusion_matrix_no_yes": [
      [
        7352,
        622
      ],
      [
        517,
        532
      ]
    ]
  },
  "holdout_threshold_05": {
    "ap": 0.45148740204

## 이번 실행의 결과 요약
- 선택 모델: **HGB_regularized**
- 개발 3-fold AP: **0.4482 ± 0.0040** (모델 선택에 사용)
- 홀드아웃 AP: **0.4515**, ROC-AUC: **0.7899**
- OOF 임계값 0.2036: 홀드아웃 F1 **0.4830**, 정확도 **0.8738**
- 학습·분석 시간: **23.6초** (패키지 설치와 최초 import 제외)

이 요약은 전달 시점의 실행 결과입니다. 재실행 결과는 각 셀 출력과 run_summary.json을 확인하세요.
전체 설명은 RESULTS_KO.txt에 있습니다.